# Задание 1. Обучение нейронной сети на датасете MNIST и её бюджет на устройстве

## 1. Используйте датасет MNIST через библиотеку torchvision (https://pytorch.org/vision/stable/generated/torchvision.datasets.MNIST.html).


Скачаю датасет: 

In [3]:
import torch
import torchvision
from torchvision.datasets import MNIST

In [4]:
dataset = MNIST(download=False, root='MNIST/')

In [5]:
dataset[0][0]

In [6]:
dataset[0][1]

5

In [7]:
dataset[20][0]

In [8]:
dataset[20][1]

4

## 2.	Создайте две полносвязные нейронные сети для задачи классификации: первая с двумя слоями, вторая с тремя. Число нейронов в скрытых слоях выберите в порядке сотен, в качестве функции активации возьмите ReLU.

Немного выпендрится 😈:

In [11]:
print("Количество GPU:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

Количество GPU: 8
GPU 0: NVIDIA A100-SXM4-80GB
GPU 1: NVIDIA A100-SXM4-80GB
GPU 2: NVIDIA A100-SXM4-80GB
GPU 3: NVIDIA A100-SXM4-80GB
GPU 4: NVIDIA A100-SXM4-80GB
GPU 5: NVIDIA A100-SXM4-80GB
GPU 6: NVIDIA A100-SXM4-80GB
GPU 7: NVIDIA A100-SXM4-80GB


In [9]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./MNIST/",
    train=True,
    transform=transform,
)

test_dataset = datasets.MNIST(
    root="./MNIST/",
    train=False,
    transform=transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
)

In [10]:
train_dataset[0][0].shape

# значит размеры фоток 28 на 28, учтем это для посторения сети

torch.Size([1, 28, 28])

In [11]:
class NNTwoLayer(nn.Module):
    def __init__(self):
        super().__init__()
        # вытягиваем в вектор всю картинку
        self.flatten = nn.Flatten()
        self.first_layer = nn.Linear(28 * 28, 128 )
        self.activation = nn.ReLU()
        self.second_layer = nn.Linear(128 , 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.first_layer(x)
        x = self.activation(x)
        x = self.second_layer(x)

        return x


class NNThreeLayer(nn.Module):
    def __init__(self):
        super().__init__()

        self.flatten = nn.Flatten()
        self.first_layer = nn.Linear(28 * 28, 128 )
        self.activation = nn.ReLU()
        self.second_layer = nn.Linear(128 , 128 )
        self.third_layer = nn.Linear(128 , 10)



    def forward(self, x):
        x = self.flatten(x)
        x = self.first_layer(x)
        x = self.activation(x)
        x = self.second_layer(x)
        x = self.activation(x)
        x = self.third_layer(x)

        return x    

In [4]:
# тест
model_1 = NNTwoLayer()
model_2 = NNThreeLayer()
images = torch.randn(64, 1, 28, 28)
logits_1 = model_1(images)
logits_2 = model_2(images)
logits_1.shape, logits_2.shape

(torch.Size([64, 10]), torch.Size([64, 10]))

In [5]:
# вот такой будет батч 0 при обучении модели
list(train_loader)[0]

[tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]]],
 
 
         [[[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]]],
 
 
         [[[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]]],
 
 
         ...,
 
 
         [[[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ..

## 3.	(3 балла) Обучите обе модели и занесите значения метрики Accuracy и функции потерь в TensorBoard. Сравните результаты и ответьте письменно: влияет ли число слоёв на качество и почему вы наблюдаете именно такую картину.

In [12]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model_1 = NNTwoLayer().to(device)
model_2 = NNThreeLayer().to(device)

In [7]:
loss_fn = nn.CrossEntropyLoss()

optimizer_1 = torch.optim.Adam(
    model_1.parameters(),
    lr=0.001,
)

optimizer_2 = torch.optim.Adam(
    model_2.parameters(),
    lr=0.001,
)

In [8]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter("runs/mnist_experiment_1")

In [9]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter("runs/two_models")

epochs = 5

for epoch in range(epochs):
    model_1.train()
    model_2.train()

    total_loss_1 = 0
    total_loss_2 = 0

    correct_1 = 0
    correct_2 = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer_1.zero_grad()

        logits_1 = model_1(images)
        loss_1 = loss_fn(logits_1, labels)

        loss_1.backward()
        optimizer_1.step()




        optimizer_2.zero_grad()

        logits_2 = model_2(images)
        loss_2 = loss_fn(logits_2, labels)

        loss_2.backward()
        optimizer_2.step()

        total_loss_1 += loss_1.item()
        total_loss_2 += loss_2.item()

        predictions_1 = logits_1.argmax(dim=1)
        predictions_2 = logits_2.argmax(dim=1)

        correct_1 += (predictions_1 == labels).sum().item()
        correct_2 += (predictions_2 == labels).sum().item()

        total += labels.size(0)

    average_loss_1 = total_loss_1 / len(train_loader)
    average_loss_2 = total_loss_2 / len(train_loader)

    accuracy_1 = correct_1 / total
    accuracy_2 = correct_2 / total

    # Две линии на одном графике
    writer.add_scalars(
        "Loss/train",
        {
            "model_1": average_loss_1,
            "model_2": average_loss_2,
        },
        epoch,
    )

    writer.add_scalars(
        "Accuracy/train",
        {
            "model_1": accuracy_1,
            "model_2": accuracy_2,
        },
        epoch,
    )

    print(
        f"Epoch {epoch + 1}/{epochs}\n"
        f"  model_1: loss={average_loss_1:.4f}, "
        f"accuracy={accuracy_1:.2%}\n"
        f"  model_2: loss={average_loss_2:.4f}, "
        f"accuracy={accuracy_2:.2%}"
    )

writer.close()

Epoch 1/5
  model_1: loss=0.3500, accuracy=90.46%
  model_2: loss=0.3132, accuracy=91.12%
Epoch 2/5
  model_1: loss=0.1580, accuracy=95.38%
  model_2: loss=0.1257, accuracy=96.21%
Epoch 3/5
  model_1: loss=0.1098, accuracy=96.73%
  model_2: loss=0.0863, accuracy=97.32%
Epoch 4/5
  model_1: loss=0.0832, accuracy=97.51%
  model_2: loss=0.0646, accuracy=97.98%
Epoch 5/5
  model_1: loss=0.0654, accuracy=98.07%
  model_2: loss=0.0506, accuracy=98.41%


In [11]:
writer.flush()
writer.close()

Что-то я не разобрался, как выводить TensorBoard в ячейки в VS Code, апоэтому прикрепляю скринами. 

Но можно будет проверить вроде так (логи из rums/ я загружу в репозиторий):
```python
%load_ext tensorboard
%tensorboard --logdir runs
```

![Accuracy в TensorBoard](images/1.png)

![Loss в TensorBoard](images/2.png)

### Ответ на вопрос 3:


Качество и скорость падения loss зависят от числа параметров!

Почему? Потому что модели проще запомнить/выявить закономерности в данных. Так как параметров больше, можно сказать, что у модели больше места, куда можно «записать» свои предположения.

Для числа слоёв работает то же самое правило. Но вот со свёртками немного интереснее: с увеличением числа слоёв в CNN увеличивается пятно, на которое смотрит модель, aka разрешающая способность модели. Она может захватить больший объём исходной картинки и увидеть знакомые паттерны.

## 4.	(2 балла) Реализуйте визуализацию в TensorBoard: в каждом батче валидационной выборки возьмите по 10 объектов и выведите изображение вместе с подписью — меткой предсказанного класса.

In [17]:
import torch
import matplotlib.pyplot as plt


def log_batch_predictions(
    writer,
    images,
    predictions,
    model_name,
    global_step,
):
    # В последнем батче может оказаться меньше 10 изображений
    count = min(10, images.size(0))

    images = images[:count].detach().cpu()
    predictions = predictions[:count].detach().cpu()

    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    axes = axes.flatten()

    for index, ax in enumerate(axes):
        if index < count:
            # MNIST имеет форму [1, 28, 28]
            image = images[index].squeeze(0)

            ax.imshow(image, cmap="gray")
            ax.set_title(
                f"Prediction: {predictions[index].item()}"
            )

        ax.axis("off")

    fig.suptitle(f"{model_name}: validation predictions")
    fig.tight_layout()

    writer.add_figure(
        tag=f"Predictions/{model_name}",
        figure=fig,
        global_step=global_step,
    )

    # Освобождаем память
    plt.close(fig)

In [ ]:
# Валидация одной модели
def validate(
    model,
    validation_loader,
    loss_fn,
    device,
    writer,
    model_name,
    epoch,
):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_index, (images, labels) in enumerate(validation_loader):
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = loss_fn(logits, labels)

            predictions = logits.argmax(dim=1)

            total_loss += loss.item()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

            # Уникальный шаг для каждого батча
            global_step = (
                epoch * len(validation_loader)
                + batch_index
            )

            # Записываем 10 изображений текущего батча
            log_batch_predictions(
                writer=writer,
                images=images,
                predictions=predictions,
                model_name=model_name,
                global_step=global_step,
            )

    average_loss = total_loss / len(validation_loader)
    accuracy = correct / total

    return average_loss, accuracy

In [19]:
val_loss_1, val_accuracy_1 = validate(
    model=model_1,
    validation_loader=test_loader,
    loss_fn=loss_fn,
    device=device,
    writer=writer,
    model_name="model_1",
    epoch=epoch,
)

val_loss_2, val_accuracy_2 = validate(
    model=model_2,
    validation_loader=test_loader,
    loss_fn=loss_fn,
    device=device,
    writer=writer,
    model_name="model_2",
    epoch=epoch,
)

In [20]:
writer.add_scalars(
    "Validation/Loss",
    {
        "model_1": val_loss_1,
        "model_2": val_loss_2,
    },
    epoch,
)

writer.add_scalars(
    "Validation/Accuracy",
    {
        "model_1": val_accuracy_1,
        "model_2": val_accuracy_2,
    },
    epoch,
)

writer.flush()

In [21]:
writer.close()

### Ответ на 4 задание:

![Predict от меньшей на 2 слоя модели](images/model_3.png)

![Predict от большей модели](images/model_4.png)

<!--  -->

## 5.	(1 балл) Возьмите реализацию LeNet и обучите её на той же задаче. Сравните качество со своими полносвязными сетями

Вот реализация LeNet. Но исходная LeNet начиналась с 32*32 изображения, а в MNIST 28*28. Чуть изменяться размерности

![LeNet](images/lenet.png)

In [13]:
import torch
from torch import nn

class LeNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            # [B, 1, 28, 28] → [B, 6, 24, 24]
            nn.Conv2d(
                in_channels=1,
                out_channels=6,
                kernel_size=5,
            ),
            nn.ReLU(),

            # [B, 6, 24, 24] → [B, 6, 12, 12]
            nn.AvgPool2d(kernel_size=2, stride=2),

            # [B, 6, 12, 12] → [B, 16, 8, 8]
            nn.Conv2d(
                in_channels=6,
                out_channels=16,
                kernel_size=5,
            ),
            nn.ReLU(),

            # [B, 16, 8, 8] → [B, 16, 4, 4]
            nn.AvgPool2d(kernel_size=2, stride=2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 4 * 4, 120),
            nn.ReLU(),
            nn.Linear(120, 84),
            nn.ReLU(),
            nn.Linear(84, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [14]:
lenet = LeNet().to(device)

test_images = torch.randn(64, 1, 28, 28).to(device)
test_logits = lenet(test_images)

print(test_logits.shape)
# torch.Size([64, 10])

torch.Size([64, 10])


In [40]:
optimizer_lenet = torch.optim.Adam(
    lenet.parameters(),
    lr=0.001,
)

In [41]:
def train_one_epoch(
    model,
    train_loader,
    loss_fn,
    optimizer,
    device,
):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images)
        loss = loss_fn(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    average_loss = total_loss / len(train_loader)
    accuracy = correct / total

    return average_loss, accuracy

In [42]:
epochs = 5

lenet_history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
}

for epoch in range(epochs):
    train_loss, train_accuracy = train_one_epoch(
        model=lenet,
        train_loader=train_loader,
        loss_fn=loss_fn,
        optimizer=optimizer_lenet,
        device=device,
    )

    val_loss, val_accuracy = validate(
        model=lenet,
        validation_loader=test_loader,
        loss_fn=loss_fn,
        device=device,
        writer=writer,
        model_name="LeNet",
        epoch=epoch,
    )

    lenet_history["train_loss"].append(train_loss)
    lenet_history["train_accuracy"].append(train_accuracy)
    lenet_history["val_loss"].append(val_loss)
    lenet_history["val_accuracy"].append(val_accuracy)

    writer.add_scalar(
        "Loss/train/LeNet",
        train_loss,
        epoch,
    )

    writer.add_scalar(
        "Accuracy/train/LeNet",
        train_accuracy,
        epoch,
    )

    writer.add_scalar(
        "Loss/validation/LeNet",
        val_loss,
        epoch,
    )

    writer.add_scalar(
        "Accuracy/validation/LeNet",
        val_accuracy,
        epoch,
    )

    writer.flush()

    print(
        f"Epoch {epoch + 1}/{epochs}: "
        f"train loss={train_loss:.4f}, "
        f"train accuracy={train_accuracy:.2%}, "
        f"val loss={val_loss:.4f}, "
        f"val accuracy={val_accuracy:.2%}"
    )

Epoch 1/5: train loss=0.3818, train accuracy=88.09%, val loss=0.1189, val accuracy=96.05%
Epoch 2/5: train loss=0.1140, train accuracy=96.53%, val loss=0.0915, val accuracy=97.11%
Epoch 3/5: train loss=0.0792, train accuracy=97.58%, val loss=0.0581, val accuracy=98.22%
Epoch 4/5: train loss=0.0610, train accuracy=98.11%, val loss=0.0441, val accuracy=98.55%
Epoch 5/5: train loss=0.0494, train accuracy=98.46%, val loss=0.0444, val accuracy=98.50%


### Ответ на 5 задание:

Предсказуемо, LeNet оказалась лучше, чем полносвязные сети

train accuracy: 98.41-> 98.46%

val accuracy: 97.6 -> 98.50%

## 6.	(2 балла) Соберите для всех трёх обученных моделей таблицу бюджета — так же, как мы делали на занятии 3. Для каждой модели укажите:

−	число параметров и их размер в байтах во float32;

−	пиковую активацию при батче из одного объекта, то есть максимум по слоям от суммы «вход плюс выход»;

−	число MAC-операций.



In [44]:
model_1

NNTwoLayer(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (first_layer): Linear(in_features=784, out_features=128, bias=True)
  (activation): ReLU()
  (second_layer): Linear(in_features=128, out_features=10, bias=True)
)

In [45]:
model_2

NNThreeLayer(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (first_layer): Linear(in_features=784, out_features=128, bias=True)
  (activation): ReLU()
  (second_layer): Linear(in_features=128, out_features=128, bias=True)
  (third_layer): Linear(in_features=128, out_features=10, bias=True)
)

In [46]:
lenet

LeNet(
  (features): Sequential(
    (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
    (1): ReLU()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    (4): ReLU()
    (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=256, out_features=120, bias=True)
    (2): ReLU()
    (3): Linear(in_features=120, out_features=84, bias=True)
    (4): ReLU()
    (5): Linear(in_features=84, out_features=10, bias=True)
  )
)

In [15]:
# функции кончено подсказал GPT

def tensor_numel(value):
    """Количество элементов во вложенной структуре тензоров."""
    if isinstance(value, torch.Tensor):
        return value.numel()

    if isinstance(value, (tuple, list)):
        return sum(tensor_numel(item) for item in value)

    if isinstance(value, dict):
        return sum(tensor_numel(item) for item in value.values())

    return 0


def calculate_model_budget(model, input_shape):
    """
    input_shape указывается без batch:
    MNIST: (1, 28, 28)
    """

    # -------------------------
    # Параметры модели
    # -------------------------

    total_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    trainable_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    # По условию считаем, что каждый параметр — float32
    parameter_bytes = total_parameters * 4

    # -------------------------
    # Активации и MAC
    # -------------------------

    peak_activation_elements = 0
    peak_activation_layer = None
    total_mac = 0

    hooks = []

    def forward_hook(module, inputs, output):
        nonlocal peak_activation_elements
        nonlocal peak_activation_layer
        nonlocal total_mac

        input_elements = tensor_numel(inputs)
        output_elements = tensor_numel(output)

        # По условию: вход слоя + выход слоя
        activation_elements = input_elements + output_elements

        if activation_elements > peak_activation_elements:
            peak_activation_elements = activation_elements
            peak_activation_layer = module.__class__.__name__

        # MAC для полносвязного слоя
        if isinstance(module, nn.Linear):
            # Для каждого выходного элемента:
            # in_features умножений-сложений
            total_mac += output.numel() * module.in_features

        # MAC для свёрточного слоя
        elif isinstance(module, nn.Conv2d):
            kernel_height, kernel_width = module.kernel_size

            operations_per_output = (
                module.in_channels
                // module.groups
                * kernel_height
                * kernel_width
            )

            total_mac += output.numel() * operations_per_output

    # Ставим hooks только на конечные слои,
    # а не на целые Sequential
    for module in model.modules():
        is_leaf_layer = len(list(module.children())) == 0

        if is_leaf_layer:
            hook = module.register_forward_hook(forward_hook)
            hooks.append(hook)

    # Определяем устройство модели
    try:
        device = next(model.parameters()).device
    except StopIteration:
        device = torch.device("cpu")

    example_input = torch.zeros(
        (1, *input_shape),
        dtype=torch.float32,
        device=device,
    )

    model.eval()

    try:
        with torch.no_grad():
            model(example_input)
    finally:
        # Hooks обязательно нужно удалить
        for hook in hooks:
            hook.remove()

    peak_activation_bytes = peak_activation_elements * 4

    return {
        "Модель": model.__class__.__name__,
        "Параметры": total_parameters,
        "Обучаемые параметры": trainable_parameters,
        "Параметры, байт (float32)": parameter_bytes,
        "Пиковая активация, элементов": peak_activation_elements,
        "Пиковая активация, байт": peak_activation_bytes,
        "Слой пиковой активации": peak_activation_layer,
        "MAC": total_mac,
    }

In [ ]:
import pandas as pd



models = {
    "Полносвязная модель 1": model_1,
    "Полносвязная модель 2": model_2,
    "LeNet": lenet,
}

rows = []

for model_name, model in models.items():
    budget = calculate_model_budget(
        model=model,
        input_shape=(1, 28, 28),
    )

    budget["Модель"] = model_name
    rows.append(budget)

budget_table = pd.DataFrame(rows)

budget_table

,Модель,Параметры,Обучаемые параметры,"Параметры, байт (float32)","Пиковая активация, элементов","Пиковая активация, байт",Слой пиковой активации,MAC
0,Полносвязная модель 1,101770,101770,407080,1568,6272,Flatten,101632
1,Полносвязная модель 2,118282,118282,473128,1568,6272,Flatten,118016
2,LeNet,44426,44426,177704,6912,27648,ReLU,281640


Себе напоминаю:

Для Linear:

MAC = количество выходных элементов × число входов

Для Conv2d:

MAC = число выходных элементов
      × входные каналы
      × высота ядра
      × ширина ядра


## 7.	(2 балла) Ответьте на два вопроса, опираясь на таблицу из пункта 6:
−	какая из трёх моделей поместится во Flash Raspberry Pi Pico 2 (4 МБ) и уложится в его оперативную память (520 КБ), если веса квантизовать в int8, то есть уменьшить вчетверо;

−	у какой модели узкое место в памяти под параметры, а у какой — в памяти под активации, и что из этого следует, если модель придётся ужимать.


In [17]:
budget_table['Параметры, байт (int8)'] = budget_table['Параметры, байт (float32)']/4

In [20]:
flash_limit = 4 * 1024 * 1024
ram_limit = 520 * 1024

budget_table["Помещается во Flash"] = (
    budget_table["Параметры"] <= flash_limit
)

budget_table["Помещается в RAM"] = (
    budget_table["Пиковая активация, байт"] <= ram_limit
)

In [21]:
budget_table[['Модель', 'Параметры, байт (int8)', 'Пиковая активация, байт', 'Помещается во Flash', "Помещается в RAM"]]

,Модель,"Параметры, байт (int8)","Пиковая активация, байт",Помещается во Flash,Помещается в RAM
0,Полносвязная модель 1,101770.0,6272,True,True
1,Полносвязная модель 2,118282.0,6272,True,True
2,LeNet,44426.0,27648,True,True


### первый вопрос: 

Все поместиться с большим запасом. Можно аж почти 19 этих LeNet одновремено запустить и RAM выдержит (без учеба ОС)

### Второй вопрос:

У LeNen узкое место - это пиковая активация, у нее одна и таже свертка применяется много раз. Из-за этого растет потребление RAM, а кол-во параметров не много.

У FNN наоборот, параметров много, но зато операции вычислений меньше в моменте. Меньше держать в памяти чисел надо.